In [ ]:
# i want to make sure verything that is going to be porcess goes into
# the right folder , it will save me time once i have to submit the code

from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

# Main project folders
DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"
NOTEBOOKS = DATA_RAW / "notebooks"

# Create folders if they do not already exist
for folder in [DATA_PROCESSED, FIGURES, TABLES, NOTEBOOKS]:
    folder.mkdir(parents=True, exist_ok=True)

# H2S input and planned output files
h2s_input_path = DATA_RAW / "h2s_clean_2024_2025.csv"

h2s_events_output_path = DATA_PROCESSED / "h2s_events_clean_2024_2025.csv"
h2s_daily_station_output_path = DATA_PROCESSED / "h2s_daily_station_summary_2024_2025.csv"

# Confirm setup
print("Project folders ready:")
print("DATA_RAW:", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("FIGURES:", FIGURES)
print("TABLES:", TABLES)
print("NOTEBOOKS:", NOTEBOOKS)

print("\nH2S input file check:")
print("Input path:", h2s_input_path)
print("File exists:", h2s_input_path.exists())

print("\nPlanned output files:")
print(h2s_events_output_path)
print(h2s_daily_station_output_path)

Mounted at /content/drive
Project folders ready:
DATA_RAW: /content/drive/MyDrive/STAT596/Project596_datafiles
DATA_PROCESSED: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data
FIGURES: /content/drive/MyDrive/STAT596/Project596_datafiles/figures
TABLES: /content/drive/MyDrive/STAT596/Project596_datafiles/tables
NOTEBOOKS: /content/drive/MyDrive/STAT596/Project596_datafiles/notebooks

H2S input file check:
Input path: /content/drive/MyDrive/STAT596/Project596_datafiles/h2s_clean_2024_2025.csv
File exists: True

Planned output files:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_events_clean_2024_2025.csv
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_daily_station_summary_2024_2025.csv


In [ ]:
# data inspection stage

h2s_raw = pd.read_csv(h2s_input_path)

# Basic structure
print("H2S raw dataset loaded")
print("=" * 60)
print("Rows:", h2s_raw.shape[0])
print("Columns:", h2s_raw.shape[1])

print("\nColumn names:")
for col in h2s_raw.columns:
    print("-", col)

print("\nData types:")
print(h2s_raw.dtypes)

print("\nMissing values by column:")
print(h2s_raw.isna().sum())

print("\nDuplicate rows:")
print(h2s_raw.duplicated().sum())

print("\nFirst 5 rows:")
display(h2s_raw.head())

H2S raw dataset loaded
Rows: 597
Columns: 22

Column names:
- datetime
- date
- time
- hour
- year
- month
- station_name
- location
- h2s_ppb_original
- h2s_ppb_clean
- unit
- h2s_level
- exceedance_30ppb
- exceedance_200ppb
- nighttime_flag
- analysis_period
- review_status
- negative_value_flag
- missing_h2s_flag
- source
- source_note
- notes

Data types:
datetime                object
date                    object
time                    object
hour                     int64
year                     int64
month                   object
station_name            object
location                object
h2s_ppb_original       float64
h2s_ppb_clean          float64
unit                    object
h2s_level               object
exceedance_30ppb         int64
exceedance_200ppb        int64
nighttime_flag           int64
analysis_period         object
review_status           object
negative_value_flag      int64
missing_h2s_flag         int64
source                  object
source_note       

,datetime,date,time,hour,year,month,station_name,location,h2s_ppb_original,h2s_ppb_clean,...,exceedance_30ppb,exceedance_200ppb,nighttime_flag,analysis_period,review_status,negative_value_flag,missing_h2s_flag,source,source_note,notes
0,2024-09-12 11:00:00,2024-09-12,11:00:00 AM,11,2024,2024-09,San Ysidro - Fire Station #29,SAN YSIDRO,NaN,NaN,...,0,0,0,Immediate post-emergency,missing_visible,0,1,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
1,2024-09-25 14:00:00,2024-09-25,2:00:00 PM,14,2024,2024-09,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,0.1,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
2,2024-09-25 15:00:00,2024-09-25,3:00:00 PM,15,2024,2024-09,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,0.1,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
3,2024-09-25 16:00:00,2024-09-25,4:00:00 PM,16,2024,2024-09,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,0.1,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
4,2024-09-25 17:00:00,2024-09-25,5:00:00 PM,17,2024,2024-09,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,0.1,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction


In [ ]:
# Identifies the H2S time period, monitoring locations, value ranges, and QA flags before
# creating final cleaned outputs.

h2s_check = h2s_raw.copy()

# Convert date/time fields for inspection only
h2s_check["datetime_parsed"] = pd.to_datetime(h2s_check["datetime"], errors="coerce")
h2s_check["date_parsed"] = pd.to_datetime(h2s_check["date"], errors="coerce")

print("H2S key field profile")
print("=" * 60)

print("Rows:", len(h2s_check))
print("Columns:", h2s_check.shape[1])

print("\nDatetime parsing check:")
print("Missing datetime_parsed:", h2s_check["datetime_parsed"].isna().sum())
print("Missing date_parsed:", h2s_check["date_parsed"].isna().sum())

print("\nDate/time coverage:")
print("Start datetime:", h2s_check["datetime_parsed"].min())
print("End datetime:", h2s_check["datetime_parsed"].max())
print("Unique dates:", h2s_check["date_parsed"].nunique())
print("Unique hours:", sorted(h2s_check["hour"].dropna().unique()))

print("\nMonitoring locations:")
print(h2s_check["location"].value_counts(dropna=False))

print("\nStation names:")
print(h2s_check["station_name"].value_counts(dropna=False))

print("\nH2S level counts:")
print(h2s_check["h2s_level"].value_counts(dropna=False))

print("\nReview status counts:")
print(h2s_check["review_status"].value_counts(dropna=False))

print("\nQA flag totals:")
qa_cols = [
    "exceedance_30ppb",
    "exceedance_200ppb",
    "nighttime_flag",
    "negative_value_flag",
    "missing_h2s_flag"
]
print(h2s_check[qa_cols].sum())

print("\nH2S original value summary:")
display(h2s_check["h2s_ppb_original"].describe())

print("\nH2S clean value summary:")
display(h2s_check["h2s_ppb_clean"].describe())

print("\nCross-tab: h2s_level by location")
display(pd.crosstab(h2s_check["location"], h2s_check["h2s_level"], margins=True))

print("\nCross-tab: review_status by h2s_level")
display(pd.crosstab(h2s_check["review_status"], h2s_check["h2s_level"], margins=True))


H2S key field profile
Rows: 597
Columns: 24

Datetime parsing check:
Missing datetime_parsed: 0
Missing date_parsed: 0

Date/time coverage:
Start datetime: 2024-09-12 11:00:00
End datetime: 2025-08-26 04:00:00
Unique dates: 26
Unique hours: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]

Monitoring locations:
location
SAN YSIDRO      235
NESTOR - BES    219
IB CIVIC CTR    143
Name: count, dtype: int64

Station names:
station_name
San Ysidro - Fire Station #29       235
Nestor - Berry Elementary School    219
Imperial Beach Civic Center         143
Name: count, dtype: int64

H2S level counts:
h2s_level
Green (0-4.9 ppB)      319
Missing/Not visible    149
Yellow (5-29.9 ppB)    118
Orange (30+ ppB)        11
Na

,h2s_ppb_original
count,513.000000
mean,5.641326
std,20.999592
min,-18.000000
25%,0.500000
50%,1.900000
75%,5.000000
max,295.600000



H2S clean value summary:


,h2s_ppb_clean
count,448.000000
mean,6.883705
std,22.150776
min,0.000000
25%,0.800000
50%,2.950000
75%,5.950000
max,295.600000



Cross-tab: h2s_level by location


h2s_level,Green (0-4.9 ppB),Missing/Not visible,Orange (30+ ppB),Yellow (5-29.9 ppB),All
location,,,,,
IB CIVIC CTR,111,13,0,19,143
NESTOR - BES,94,58,7,60,219
SAN YSIDRO,114,78,4,39,235
All,319,149,11,118,597



Cross-tab: review_status by h2s_level


h2s_level,Green (0-4.9 ppB),Missing/Not visible,Orange (30+ ppB),Yellow (5-29.9 ppB),All
review_status,,,,,
missing_visible,0,16,0,0,16
numeric,95,62,1,16,174
screenshot_numeric,199,3,8,77,287
screenshot_numeric_partial_row,1,0,0,0,1
screenshot_verified,24,0,2,25,51
source_cell_blank_in_screenshot,0,68,0,0,68
All,319,149,11,118,597


In [ ]:
# so far i believe theres is only around 448 records that are useful for the
# project. This is the dataset where data was limited and never got a response
# from the city
#so now it has to be Standardizes key fields ect

h2s_events = h2s_raw.copy()

# Parse date and datetime fields
h2s_events["datetime_local"] = pd.to_datetime(h2s_events["datetime"], errors="coerce")
h2s_events["date"] = pd.to_datetime(h2s_events["date"], errors="coerce").dt.date

# Standardize station/location fields
h2s_events["location_clean"] = h2s_events["location"].astype(str).str.strip().str.upper()
h2s_events["station_name_clean"] = h2s_events["station_name"].astype(str).str.strip()

# Use cleaned numeric H2S concentration for analysis
h2s_events["h2s_ppb"] = pd.to_numeric(h2s_events["h2s_ppb_clean"], errors="coerce")

# Flag records usable for numeric analysis
h2s_events["valid_h2s_record"] = h2s_events["h2s_ppb"].notna()

# Project event thresholds
h2s_events["elevated_h2s_event"] = h2s_events["valid_h2s_record"] & (h2s_events["h2s_ppb"] >= 5)
h2s_events["orange_h2s_event"] = h2s_events["valid_h2s_record"] & (h2s_events["h2s_ppb"] >= 30)

# Recalculate simplified level labels from cleaned H2S value
h2s_events["h2s_level_clean"] = np.select(
    [
        h2s_events["h2s_ppb"].isna(),
        h2s_events["h2s_ppb"] < 5,
        (h2s_events["h2s_ppb"] >= 5) & (h2s_events["h2s_ppb"] < 30),
        h2s_events["h2s_ppb"] >= 30
    ],
    [
        "Missing/Not visible",
        "Green (0-4.9 ppB)",
        "Yellow (5-29.9 ppB)",
        "Orange (30+ ppB)"
    ],
    default="Check"
)

# Add time fields useful for later temporal summaries
h2s_events["year"] = h2s_events["datetime_local"].dt.year
h2s_events["month"] = h2s_events["datetime_local"].dt.to_period("M").astype(str)
h2s_events["hour"] = h2s_events["datetime_local"].dt.hour

# Keep a clean, organized column order
h2s_keep_cols = [
    "datetime_local",
    "date",
    "year",
    "month",
    "hour",
    "station_name_clean",
    "location_clean",
    "h2s_ppb",
    "h2s_level_clean",
    "elevated_h2s_event",
    "orange_h2s_event",
    "valid_h2s_record",
    "h2s_ppb_original",
    "h2s_ppb_clean",
    "h2s_level",
    "exceedance_30ppb",
    "exceedance_200ppb",
    "nighttime_flag",
    "analysis_period",
    "review_status",
    "negative_value_flag",
    "missing_h2s_flag",
    "source",
    "source_note",
    "notes"
]

h2s_events = h2s_events[h2s_keep_cols].sort_values(
    ["datetime_local", "location_clean"]
).reset_index(drop=True)

print("Cleaned H2S event dataset created")
print("=" * 60)
print("Rows:", h2s_events.shape[0])
print("Columns:", h2s_events.shape[1])
print("Valid numeric H2S records:", h2s_events["valid_h2s_record"].sum())
print("Missing/not usable H2S records:", (~h2s_events["valid_h2s_record"]).sum())
print("Elevated H2S events >= 5 ppb:", h2s_events["elevated_h2s_event"].sum())
print("Orange H2S events >= 30 ppb:", h2s_events["orange_h2s_event"].sum())

print("\nClean H2S level counts:")
print(h2s_events["h2s_level_clean"].value_counts(dropna=False))

print("\nPreview:")
display(h2s_events.head())

Cleaned H2S event dataset created
Rows: 597
Columns: 25
Valid numeric H2S records: 448
Missing/not usable H2S records: 149
Elevated H2S events >= 5 ppb: 129
Orange H2S events >= 30 ppb: 11

Clean H2S level counts:
h2s_level_clean
Green (0-4.9 ppB)      319
Missing/Not visible    149
Yellow (5-29.9 ppB)    118
Orange (30+ ppB)        11
Name: count, dtype: int64

Preview:


,datetime_local,date,year,month,hour,station_name_clean,location_clean,h2s_ppb,h2s_level_clean,elevated_h2s_event,...,exceedance_30ppb,exceedance_200ppb,nighttime_flag,analysis_period,review_status,negative_value_flag,missing_h2s_flag,source,source_note,notes
0,2024-09-12 11:00:00,2024-09-12,2024,2024-09,11,San Ysidro - Fire Station #29,SAN YSIDRO,NaN,Missing/Not visible,False,...,0,0,0,Immediate post-emergency,missing_visible,0,1,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
1,2024-09-25 14:00:00,2024-09-25,2024,2024-09,14,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,Green (0-4.9 ppB),False,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
2,2024-09-25 15:00:00,2024-09-25,2024,2024-09,15,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,Green (0-4.9 ppB),False,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
3,2024-09-25 16:00:00,2024-09-25,2024,2024-09,16,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,Green (0-4.9 ppB),False,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction
4,2024-09-25 17:00:00,2024-09-25,2024,2024-09,17,San Ysidro - Fire Station #29,SAN YSIDRO,0.1,Green (0-4.9 ppB),False,...,0,0,0,Immediate post-emergency,numeric,0,0,SDAPCD Power BI dashboard screen recording,Manually extracted from SDAPCD public Power BI...,from initial screen-recording extraction


In [ ]:
# summarizing the data by location , month and analysis period
# this will help with matching it with the other datasets

print("Cleaned H2S QA summary")
print("=" * 60)

print("Date/time coverage:")
print("Start datetime:", h2s_events["datetime_local"].min())
print("End datetime:", h2s_events["datetime_local"].max())
print("Unique dates:", h2s_events["date"].nunique())

print("\nRecords by location:")
display(
    h2s_events.groupby("location_clean")
    .agg(
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        missing_or_invalid_records=("valid_h2s_record", lambda x: (~x).sum()),
        mean_h2s_ppb=("h2s_ppb", "mean"),
        median_h2s_ppb=("h2s_ppb", "median"),
        max_h2s_ppb=("h2s_ppb", "max"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum")
    )
    .reset_index()
    .sort_values("elevated_h2s_events", ascending=False)
)

print("\nClean H2S levels by location:")
display(pd.crosstab(h2s_events["location_clean"], h2s_events["h2s_level_clean"], margins=True))

print("\nRecords by month:")
display(
    h2s_events.groupby("month")
    .agg(
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum"),
        max_h2s_ppb=("h2s_ppb", "max")
    )
    .reset_index()
)

print("\nRecords by analysis period:")
display(
    h2s_events.groupby("analysis_period")
    .agg(
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum"),
        max_h2s_ppb=("h2s_ppb", "max")
    )
    .reset_index()
)

Cleaned H2S QA summary
Date/time coverage:
Start datetime: 2024-09-12 11:00:00
End datetime: 2025-08-26 04:00:00
Unique dates: 26

Records by location:


,location_clean,total_records,valid_h2s_records,missing_or_invalid_records,mean_h2s_ppb,median_h2s_ppb,max_h2s_ppb,elevated_h2s_events,orange_h2s_events
1,NESTOR - BES,219,161,58,11.749068,4.0,295.6,67,7
2,SAN YSIDRO,235,157,78,4.958599,2.6,45.5,43,4
0,IB CIVIC CTR,143,130,13,3.183077,1.7,27.9,19,0



Clean H2S levels by location:


h2s_level_clean,Green (0-4.9 ppB),Missing/Not visible,Orange (30+ ppB),Yellow (5-29.9 ppB),All
location_clean,,,,,
IB CIVIC CTR,111,13,0,19,143
NESTOR - BES,94,58,7,60,219
SAN YSIDRO,114,78,4,39,235
All,319,149,11,118,597



Records by month:


,month,total_records,valid_h2s_records,elevated_h2s_events,orange_h2s_events,max_h2s_ppb
0,2024-09,26,25,2,0,10.0
1,2024-10,38,14,1,0,6.5
2,2024-11,23,9,0,0,3.7
3,2024-12,140,127,43,0,29.8
4,2025-01,126,83,18,0,13.4
5,2025-02,35,24,1,0,5.0
6,2025-04,17,8,1,0,5.0
7,2025-05,147,147,61,10,295.6
8,2025-07,32,11,2,1,207.4
9,2025-08,13,0,0,0,NaN



Records by analysis period:


,analysis_period,total_records,valid_h2s_records,elevated_h2s_events,orange_h2s_events,max_h2s_ppb
0,Immediate post-emergency,227,175,46,0,29.8
1,Recent follow-up,370,273,83,11,295.6


In [ ]:
# now lets move to daily station level summary of h2s

h2s_daily_station = (
    h2s_events
    .groupby(["date", "location_clean", "station_name_clean"], dropna=False)
    .agg(
        first_datetime_local=("datetime_local", "min"),
        last_datetime_local=("datetime_local", "max"),
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        missing_or_invalid_records=("valid_h2s_record", lambda x: (~x).sum()),
        mean_h2s_ppb=("h2s_ppb", "mean"),
        median_h2s_ppb=("h2s_ppb", "median"),
        max_h2s_ppb=("h2s_ppb", "max"),
        min_h2s_ppb=("h2s_ppb", "min"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum"),
        exceedance_200ppb_events=("exceedance_200ppb", "sum"),
        nighttime_records=("nighttime_flag", "sum")
    )
    .reset_index()
)

# Add rates based on valid numeric H2S records
h2s_daily_station["elevated_h2s_rate"] = np.where(
    h2s_daily_station["valid_h2s_records"] > 0,
    h2s_daily_station["elevated_h2s_events"] / h2s_daily_station["valid_h2s_records"],
    np.nan
)

h2s_daily_station["orange_h2s_rate"] = np.where(
    h2s_daily_station["valid_h2s_records"] > 0,
    h2s_daily_station["orange_h2s_events"] / h2s_daily_station["valid_h2s_records"],
    np.nan
)

# Add event-day flags
h2s_daily_station["any_elevated_h2s_day"] = h2s_daily_station["elevated_h2s_events"] > 0
h2s_daily_station["any_orange_h2s_day"] = h2s_daily_station["orange_h2s_events"] > 0

# Sort for readability
h2s_daily_station = h2s_daily_station.sort_values(
    ["date", "location_clean"]
).reset_index(drop=True)

print("Daily station-level H2S summary created")
print("=" * 60)
print("Rows:", h2s_daily_station.shape[0])
print("Columns:", h2s_daily_station.shape[1])
print("Date range:", h2s_daily_station["date"].min(), "to", h2s_daily_station["date"].max())
print("Unique dates:", h2s_daily_station["date"].nunique())
print("Unique stations:", h2s_daily_station["location_clean"].nunique())

print("\nDaily station elevated event totals:")
print("Valid H2S records:", h2s_daily_station["valid_h2s_records"].sum())
print("Elevated H2S events >= 5 ppb:", h2s_daily_station["elevated_h2s_events"].sum())
print("Orange H2S events >= 30 ppb:", h2s_daily_station["orange_h2s_events"].sum())

print("\nPreview:")
display(h2s_daily_station.head(10))

Daily station-level H2S summary created
Rows: 59
Columns: 20
Date range: 2024-09-12 to 2025-08-26
Unique dates: 26
Unique stations: 3

Daily station elevated event totals:
Valid H2S records: 448
Elevated H2S events >= 5 ppb: 129
Orange H2S events >= 30 ppb: 11

Preview:


,date,location_clean,station_name_clean,first_datetime_local,last_datetime_local,total_records,valid_h2s_records,missing_or_invalid_records,mean_h2s_ppb,median_h2s_ppb,max_h2s_ppb,min_h2s_ppb,elevated_h2s_events,orange_h2s_events,exceedance_200ppb_events,nighttime_records,elevated_h2s_rate,orange_h2s_rate,any_elevated_h2s_day,any_orange_h2s_day
0,2024-09-12,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-12 11:00:00,2024-09-12 11:00:00,1,0,1,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,False,False
1,2024-09-25,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-25 14:00:00,2024-09-25 23:00:00,10,10,0,2.070000,0.15,10.0,0.1,2,0,0,4,0.200000,0.0,True,False
2,2024-09-26,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-26 00:00:00,2024-09-26 14:00:00,15,15,0,0.126667,0.10,0.2,0.0,0,0,0,8,0.000000,0.0,False,False
3,2024-10-12,NESTOR - BES,Nestor - Berry Elementary School,2024-10-12 06:00:00,2024-10-12 14:00:00,9,0,9,NaN,NaN,NaN,NaN,0,0,0,2,NaN,NaN,False,False
4,2024-10-12,SAN YSIDRO,San Ysidro - Fire Station #29,2024-10-12 06:00:00,2024-10-12 14:00:00,9,9,0,1.766667,1.00,6.5,0.4,1,0,0,2,0.111111,0.0,True,False
5,2024-10-21,SAN YSIDRO,San Ysidro - Fire Station #29,2024-10-21 17:00:09,2024-10-21 17:00:09,1,1,0,0.300000,0.30,0.3,0.3,0,0,0,0,0.000000,0.0,False,False
6,2024-10-30,NESTOR - BES,Nestor - Berry Elementary School,2024-10-30 08:00:00,2024-10-30 16:00:00,9,0,9,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,False,False
7,2024-10-30,SAN YSIDRO,San Ysidro - Fire Station #29,2024-10-30 07:00:00,2024-10-30 16:00:00,10,4,6,1.800000,1.80,3.5,0.1,0,0,0,1,0.000000,0.0,False,False
8,2024-11-08,NESTOR - BES,Nestor - Berry Elementary School,2024-11-08 17:00:00,2024-11-08 17:00:00,1,0,1,NaN,NaN,NaN,NaN,0,0,0,0,NaN,NaN,False,False
9,2024-11-08,SAN YSIDRO,San Ysidro - Fire Station #29,2024-11-08 19:00:00,2024-11-08 19:00:00,1,1,0,0.200000,0.20,0.2,0.2,0,0,0,0,0.000000,0.0,False,False


In [ ]:
# create the table we can use for the report

h2s_overall_summary = pd.DataFrame({
    "metric": [
        "total_records",
        "valid_h2s_records",
        "missing_or_invalid_records",
        "unique_dates",
        "unique_locations",
        "start_datetime",
        "end_datetime",
        "mean_h2s_ppb",
        "median_h2s_ppb",
        "max_h2s_ppb",
        "elevated_h2s_events_ge_5ppb",
        "orange_h2s_events_ge_30ppb",
        "exceedance_200ppb_events",
        "nighttime_records"
    ],
    "value": [
        len(h2s_events),
        int(h2s_events["valid_h2s_record"].sum()),
        int((~h2s_events["valid_h2s_record"]).sum()),
        int(h2s_events["date"].nunique()),
        int(h2s_events["location_clean"].nunique()),
        str(h2s_events["datetime_local"].min()),
        str(h2s_events["datetime_local"].max()),
        round(h2s_events["h2s_ppb"].mean(), 3),
        round(h2s_events["h2s_ppb"].median(), 3),
        round(h2s_events["h2s_ppb"].max(), 3),
        int(h2s_events["elevated_h2s_event"].sum()),
        int(h2s_events["orange_h2s_event"].sum()),
        int(h2s_events["exceedance_200ppb"].sum()),
        int(h2s_events["nighttime_flag"].sum())
    ]
})

# Station-level H2S summary table
h2s_station_summary = (
    h2s_events
    .groupby(["location_clean", "station_name_clean"], dropna=False)
    .agg(
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        missing_or_invalid_records=("valid_h2s_record", lambda x: (~x).sum()),
        unique_dates=("date", "nunique"),
        start_datetime=("datetime_local", "min"),
        end_datetime=("datetime_local", "max"),
        mean_h2s_ppb=("h2s_ppb", "mean"),
        median_h2s_ppb=("h2s_ppb", "median"),
        max_h2s_ppb=("h2s_ppb", "max"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum"),
        exceedance_200ppb_events=("exceedance_200ppb", "sum"),
        nighttime_records=("nighttime_flag", "sum")
    )
    .reset_index()
)

# Add station-level event rates
h2s_station_summary["elevated_h2s_rate"] = np.where(
    h2s_station_summary["valid_h2s_records"] > 0,
    h2s_station_summary["elevated_h2s_events"] / h2s_station_summary["valid_h2s_records"],
    np.nan
)

h2s_station_summary["orange_h2s_rate"] = np.where(
    h2s_station_summary["valid_h2s_records"] > 0,
    h2s_station_summary["orange_h2s_events"] / h2s_station_summary["valid_h2s_records"],
    np.nan
)

h2s_station_summary = h2s_station_summary.sort_values(
    "elevated_h2s_events", ascending=False
).reset_index(drop=True)

# Daily all-station H2S summary table
h2s_daily_summary = (
    h2s_events
    .groupby("date", dropna=False)
    .agg(
        total_records=("h2s_ppb", "size"),
        valid_h2s_records=("valid_h2s_record", "sum"),
        locations_observed=("location_clean", "nunique"),
        mean_h2s_ppb=("h2s_ppb", "mean"),
        median_h2s_ppb=("h2s_ppb", "median"),
        max_h2s_ppb=("h2s_ppb", "max"),
        elevated_h2s_events=("elevated_h2s_event", "sum"),
        orange_h2s_events=("orange_h2s_event", "sum"),
        exceedance_200ppb_events=("exceedance_200ppb", "sum"),
        nighttime_records=("nighttime_flag", "sum")
    )
    .reset_index()
)

h2s_daily_summary["any_elevated_h2s_day"] = h2s_daily_summary["elevated_h2s_events"] > 0
h2s_daily_summary["any_orange_h2s_day"] = h2s_daily_summary["orange_h2s_events"] > 0

print("H2S summary tables created")
print("=" * 60)

print("\nOverall summary:")
display(h2s_overall_summary)

print("\nStation summary:")
display(h2s_station_summary)

print("\nDaily summary preview:")
display(h2s_daily_summary.head(10))

H2S summary tables created

Overall summary:


,metric,value
0,total_records,597
1,valid_h2s_records,448
2,missing_or_invalid_records,149
3,unique_dates,26
4,unique_locations,3
5,start_datetime,2024-09-12 11:00:00
6,end_datetime,2025-08-26 04:00:00
7,mean_h2s_ppb,6.884
8,median_h2s_ppb,2.95
9,max_h2s_ppb,295.6



Station summary:


,location_clean,station_name_clean,total_records,valid_h2s_records,missing_or_invalid_records,unique_dates,start_datetime,end_datetime,mean_h2s_ppb,median_h2s_ppb,max_h2s_ppb,elevated_h2s_events,orange_h2s_events,exceedance_200ppb_events,nighttime_records,elevated_h2s_rate,orange_h2s_rate
0,NESTOR - BES,Nestor - Berry Elementary School,219,161,58,22,2024-10-12 06:00:00,2025-08-26 02:00:00,11.749068,4.0,295.6,67,7,3,115,0.416149,0.043478
1,SAN YSIDRO,San Ysidro - Fire Station #29,235,157,78,23,2024-09-12 11:00:00,2025-08-26 02:00:00,4.958599,2.6,45.5,43,4,0,121,0.273885,0.025478
2,IB CIVIC CTR,Imperial Beach Civic Center,143,130,13,14,2024-12-25 18:00:00,2025-08-26 04:00:00,3.183077,1.7,27.9,19,0,0,86,0.146154,0.000000



Daily summary preview:


,date,total_records,valid_h2s_records,locations_observed,mean_h2s_ppb,median_h2s_ppb,max_h2s_ppb,elevated_h2s_events,orange_h2s_events,exceedance_200ppb_events,nighttime_records,any_elevated_h2s_day,any_orange_h2s_day
0,2024-09-12,1,0,1,NaN,NaN,NaN,0,0,0,0,False,False
1,2024-09-25,10,10,1,2.070000,0.15,10.0,2,0,0,4,True,False
2,2024-09-26,15,15,1,0.126667,0.10,0.2,0,0,0,8,False,False
3,2024-10-12,18,9,2,1.766667,1.00,6.5,1,0,0,4,True,False
4,2024-10-21,1,1,1,0.300000,0.30,0.3,0,0,0,0,False,False
5,2024-10-30,19,4,2,1.800000,1.80,3.5,0,0,0,1,False,False
6,2024-11-08,2,1,2,0.200000,0.20,0.2,0,0,0,0,False,False
7,2024-11-18,18,8,2,1.975000,1.75,3.7,0,0,0,15,False,False
8,2024-11-26,1,0,1,NaN,NaN,NaN,0,0,0,0,False,False
9,2024-11-28,2,0,2,NaN,NaN,NaN,0,0,0,0,False,False


In [ ]:
# saving process

h2s_overall_summary_path = TABLES / "h2s_overall_summary.csv"
h2s_station_summary_path = TABLES / "h2s_station_summary.csv"
h2s_daily_summary_path = TABLES / "h2s_daily_summary.csv"

# Save processed datasets
h2s_events.to_csv(h2s_events_output_path, index=False)
h2s_daily_station.to_csv(h2s_daily_station_output_path, index=False)

# Save summary tables
h2s_overall_summary.to_csv(h2s_overall_summary_path, index=False)
h2s_station_summary.to_csv(h2s_station_summary_path, index=False)
h2s_daily_summary.to_csv(h2s_daily_summary_path, index=False)

# Reload main saved files for QA
h2s_events_reloaded = pd.read_csv(h2s_events_output_path)
h2s_daily_station_reloaded = pd.read_csv(h2s_daily_station_output_path)

print("Saved H2S processed files")
print("=" * 60)

print("\nProcessed data files:")
print("H2S events:", h2s_events_output_path)
print("File exists:", h2s_events_output_path.exists())
print("Reloaded shape:", h2s_events_reloaded.shape)

print("\nDaily station summary:", h2s_daily_station_output_path)
print("File exists:", h2s_daily_station_output_path.exists())
print("Reloaded shape:", h2s_daily_station_reloaded.shape)

print("\nSummary tables:")
for path in [h2s_overall_summary_path, h2s_station_summary_path, h2s_daily_summary_path]:
    print(path)
    print("File exists:", path.exists())

print("\nReloaded H2S event QA:")
print("Rows:", len(h2s_events_reloaded))
print("Valid H2S records:", h2s_events_reloaded["valid_h2s_record"].sum())
print("Elevated H2S events >= 5 ppb:", h2s_events_reloaded["elevated_h2s_event"].sum())
print("Orange H2S events >= 30 ppb:", h2s_events_reloaded["orange_h2s_event"].sum())

print("\nReloaded daily station QA:")
print("Rows:", len(h2s_daily_station_reloaded))
print("Unique dates:", h2s_daily_station_reloaded["date"].nunique())
print("Unique stations:", h2s_daily_station_reloaded["location_clean"].nunique())

Saved H2S processed files

Processed data files:
H2S events: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_events_clean_2024_2025.csv
File exists: True
Reloaded shape: (597, 25)

Daily station summary: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_daily_station_summary_2024_2025.csv
File exists: True
Reloaded shape: (59, 20)

Summary tables:
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/h2s_overall_summary.csv
File exists: True
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/h2s_station_summary.csv
File exists: True
/content/drive/MyDrive/STAT596/Project596_datafiles/tables/h2s_daily_summary.csv
File exists: True

Reloaded H2S event QA:
Rows: 597
Valid H2S records: 448
Elevated H2S events >= 5 ppb: 129
Orange H2S events >= 30 ppb: 11

Reloaded daily station QA:
Rows: 59
Unique dates: 26
Unique stations: 3
